In [1]:
import sys
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values


In [2]:
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

SOURCE_DIR = PROJECT_ROOT / "data" / "bronze" / "open_meteo" / "forecast"

print("Forecast source directory:", SOURCE_DIR)
print("UTC now:", datetime.now(timezone.utc).isoformat())

Forecast source directory: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast
UTC now: 2026-08-12T16:01:34.416076+00:00


In [3]:
def connect():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DATABASE"),
        user=os.getenv("POSTGRES_USERNAME"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )


def get_forecast_files(directory: Path):
    return sorted(directory.rglob("*.json"))


def build_forecast_rows(path: Path) -> list[tuple]:
    path = Path(path)
    raw = pd.read_json(path)
    hourly = raw.get("hourly", {})

    if isinstance(hourly, pd.Series):
        hourly = hourly.to_dict()

    times = pd.Series(hourly.get("time", [])).tolist()
    if not times:
        return []

    field_mapping = {
        "temperature_2m": "temperature_celsius",
        "apparent_temperature": "feels_like_celsius",
        "relative_humidity_2m": "relative_humidity_pct",
        "precipitation_probability": "precipitation_probability",
        "precipitation": "precipitation_mm",
        "weather_code": "weather_code",
        "wind_speed_10m": "wind_speed_10m",
        "wind_direction_10m": "wind_direction_10m",
    }

    data = {
        "source_name": "open_meteo",
        "source_type": "forecast",
        "city_name": path.stem.rsplit("_", 1)[0],
        "ingestion_date": path.parent.name,
        "observation_time": [f"{observation_time}Z" for observation_time in times],
    }

    for source_field, target_field in field_mapping.items():
        data[target_field] = pd.Series(hourly.get(source_field, [])).tolist()

    df = pd.DataFrame(data)
    return [tuple(row) for row in df.itertuples(index=False, name=None)]

In [4]:
forecast_files = get_forecast_files(SOURCE_DIR)

all_rows = []
skipped = []

for path in forecast_files:
    rows = build_forecast_rows(path)
    if rows:
        all_rows.extend(rows)
    else:
        skipped.append(str(path))

print(f"Found {len(forecast_files)} forecast files")
print(f"Built {len(all_rows)} rows")
if skipped:
    print(f"Skipped {len(skipped)} invalid files")

Found 60 forecast files
Built 10080 rows


In [5]:
insert_sql = """
INSERT INTO silver.forecast_weather_hourly (
    source_name, 
    source_type, 
    city_name, 
    ingestion_date, 
    observation_time,
    temperature_celsius, 
    feels_like_celsius, 
    relative_humidity_pct,
    precipitation_probability, 
    precipitation_mm, 
    weather_code,
    wind_speed_10m, 
    wind_direction_10m
) VALUES %s;
"""

if all_rows:
    with connect() as conn:
        with conn.cursor() as cursor:
            execute_values(cursor, insert_sql, all_rows)
        conn.commit()
    print("Loaded rows into silver.forecast_weather_hourly")
else:
    print("Nenhuma linha para inserir.")

Loaded rows into silver.forecast_weather_hourly
